# PPO: проверки, разогрев критика и EXP-028, пилот, обучение

Спецификация — `docs/ppo.md`. Порядок для каждого сида: (1) vLLM на карте 1 (терминал); (2) проверки — один раз;
(3) разогрев: эпизоды исходной модели (≈ 1,9 ч), признаки и голова критика — для сида 0 с выбором головы (EXP-028),
для сида 1 — с уже выбранными настройками; (4) пилот (только сид 0); (5) обучение или продолжение после сбоя.
Между разделами, занимающими карту 0 (2, 3, 4, 5), лучше перезапускать ядро. **SEED** задаётся в первой ячейке.

## 1. Сервер vLLM (терминал, карта 1) — как у GRPO

```bash
source ~/vllm085/bin/activate
cd ~/Diplom
VLLM_NO_USAGE_STATS=1 VLLM_USE_V1=0 VLLM_ALLOW_RUNTIME_LORA_UPDATING=True CUDA_VISIBLE_DEVICES=1 \
nohup python -m vllm.entrypoints.openai.api_server --model t-tech/T-lite-it-2.1 --revision d125c970c553de58fcee3c937d5e4867d4a448d8 \
    --served-model-name base --dtype float16 --enable-lora --max-lora-rank 32 --max-loras 1 \
    --max-model-len 25600 --gpu-memory-utilization 0.75 --generation-config vllm --port 8000 --seed 0 \
    > ~/vllm_ppo.log 2>&1 &
grep "startup complete" ~/vllm_ppo.log
```

In [ ]:
import json, os, statistics, subprocess, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # до первого обращения к CUDA
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.training.common import model_commit, read_journal
from presentation_rl.training.ppo import PPOConfig
from presentation_rl.training.vllm_client import VLLMClient

os.environ["WIKI_PROXY"] = ""
SEED = 0  # 0 или 1
WARMUP = ROOT / "runs" / "ppo" / f"warmup_seed{SEED}"
config = PPOConfig.load(ROOT / "configs" / "algorithms" / "ppo.yaml", seed=SEED, warmup_dir=str(WARMUP))
print(config.policy_model, "коммит:", model_commit(config.policy_model, config.policy_revision), "| сид", config.seed)
client = VLLMClient(config.vllm_url)
print("vLLM", client.version(), client.models())

## 2. Проверки (один раз; карта 0 свободна)

(а) тесты: части без torch (`tests/test_ppo.py`), части с torch на крошечной модели (`tests/test_ppo_torch.py`)
и общие проверки GRPO (`tests/test_rl_torch.py`); (б) память карты 0 — полный цикл PPO дважды на эпизоде
с максимальным контекстом (проход 0, голова, два прохода политики, состояния обоих Adam).

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_ppo.py", "tests/test_ppo_torch.py",
                         "tests/test_rl_torch.py", "tests/test_rl_common.py"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout[-4000:], result.stderr[-2000:])

In [ ]:
from presentation_rl.training.ppo import memory_check

print(memory_check(config, ROOT))  # ok=False — нехватка памяти: разбор до запусков

## 3. Разогрев критика

**3.1. Эпизоды исходной модели** (vLLM, ≈ 1,9 ч): очередь разогрева со своим сидом, `warmup_steps` × 64 эпизода;
при перезапуске готовые шаги пропускаются.

In [ ]:
from presentation_rl.training.ppo import warmup_generate

warmup_generate(config, ROOT, client)

**3.2. Признаки опорной модели** (карта 0, ≈ 20 мин): равномерная выборка позиций (≤ 200 тыс.; первое и последнее
действие — всегда), `last_hidden_state` во float16 → `features.pt`.

In [ ]:
import torch
from presentation_rl.training.common import load_policy, setup_torch
from presentation_rl.training.ppo import load_warmup_episodes, warmup_features

setup_torch()
episodes = load_warmup_episodes(WARMUP)
model, _ = load_policy(config, ROOT)
start = time.monotonic()
data = warmup_features(model, episodes, config, WARMUP)
print(f"эпизодов {len(episodes)}, позиций {sum(f.shape[0] for f in data['features'])}, "
      f"{(time.monotonic() - start) / 60:.1f} мин; награда: среднее {statistics.fmean(data['rewards']):.3f}, "
      f"σ {statistics.pstdev(data['rewards']):.3f}")
del model
torch.cuda.empty_cache()

**3.3. EXP-028 — выбор головы (только сид 0; правило — `docs/ppo.md`, раздел 16, записано до проверки).**
Последний слой; линейная голова, затем MLP-256; у каждой — два lr (1e-4, 3e-4), лучший по ошибке на отложенных
темах. Основная мера — парная разность квадратов ошибок константы и критика (бутстреп по темам). MLP — только
если линейная не «пригодна». Описательно — срезы ответа, режимы, внутри заданий, среднее других ответов задания.

In [ ]:
import torch
from presentation_rl.training.ppo import POSITIONS, exp028_verdict, fit_warmup_head, split_topics
from presentation_rl.training.common import stable_seed

data = torch.load(WARMUP / "features.pt", weights_only=False)
HOLD = split_topics(data["tasks"], config.warmup_holdout, stable_seed(config.seed, "warmup-split"))
print("отложенных тем:", len(HOLD), "из", len(set(data["tasks"])))


def show(fit):
    m = fit["holdout"]
    print(f"== {fit['kind']} lr {fit['lr']}: лучшая эпоха {fit['best_epoch']}; MSE {m['all_mse']:.5f} против константы "
          f"{m['all_mse_const']:.5f}; смещение {m['all_bias']:+.4f}; EV {m['all_ev']}")
    p = m["paired"]
    print(f"   основная мера: {p['estimate']:+.5f} [{p['low']:+.5f}; {p['high']:+.5f}] по {p['topics']} темам")
    for name in POSITIONS:
        print(f"   {name:5s} MSE {m[name + '_mse']:.5f} | константа {m[name + '_mse_const']:.5f} | по режимам "
              f"{m.get(name + '_mse_mode', float('nan')):.5f} | EV внутри задания {m[name + '_ev_within']}")
    print(f"   первое действие: среднее других ответов задания {m['first_mse_group']}")
    for mode in ("search", "plain"):
        print(f"   {mode}: MSE {m.get(mode + '_all_mse')} | константа {m.get(mode + '_all_mse_const')}")
    print("   вердикт:", exp028_verdict(m))


FITS = {}
for kind in ("linear", "mlp"):
    for lr in (1e-4, 3e-4):
        FITS[(kind, lr)] = fit_warmup_head(data, config, kind, lr, HOLD)
        show(FITS[(kind, lr)])
    best = min((f for (k, _), f in FITS.items() if k == kind), key=lambda f: f["holdout"]["all_mse"])
    if exp028_verdict(best["holdout"]) == "пригоден":
        break

In [ ]:
order = [k for k in ("linear", "mlp") if any(kk == k for kk, _ in FITS)]
CHOSEN = None
for kind in order:
    best = min((f for (k, _), f in FITS.items() if k == kind), key=lambda f: f["holdout"]["all_mse"])
    verdict = exp028_verdict(best["holdout"])
    print(kind, best["lr"], best["best_epoch"], verdict)
    if verdict == "пригоден":
        CHOSEN = best
        break
if CHOSEN is None:
    print("ни одна голова не «пригодна» — решение по разделу 16 (до запуска, записать в decisions.md); "
          "ниже — по умолчанию лучшая по MSE на отложенных темах")
    CHOSEN = min(FITS.values(), key=lambda f: f["holdout"]["all_mse"])
CHOICE = {"kind": CHOSEN["kind"], "lr": CHOSEN["lr"], "epochs": CHOSEN["best_epoch"]}
(WARMUP / "exp028.json").write_text(json.dumps({"choice": CHOICE, "fits": [
    {k: v for k, v in f.items() if k not in ("head", "optimizer")} for f in FITS.values()]},
    ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("выбор:", CHOICE)

**3.4. Итоговая голова сида** — заново на всех эпизодах разогрева, выбранный вид, lr и число эпох (для сида 1 —
выбор сида 0 из `runs/ppo/warmup_seed0/exp028.json`, без перевыбора).

In [ ]:
import torch
from presentation_rl.training.ppo import fit_warmup_head, save_warmup

data = torch.load(WARMUP / "features.pt", weights_only=False)
CHOICE = json.loads((ROOT / "runs" / "ppo" / "warmup_seed0" / "exp028.json").read_text(encoding="utf-8"))["choice"]
final = fit_warmup_head(data, config, CHOICE["kind"], CHOICE["lr"], None, epochs=CHOICE["epochs"])
print(save_warmup(WARMUP, final, config, {"choice": CHOICE, "episodes": len(data["rewards"])}))

## 4. Пилот (сид 0): разогрев + 5 шагов, validation на пятом

Папка — `runs/ppo_pilot/`. Смотрю: время фаз и оценку полного запуска, пик памяти, меры критика на новых пакетах
до обновления (подтверждение EXP-028), долю отсечённых во втором проходе, максимум ρ, KL(old‖new).

In [ ]:
from dataclasses import replace
from presentation_rl.training.ppo import train

pilot = replace(config, steps=5, eval_every=5, runs_dir="runs/ppo_pilot")
STARTED = time.time()
PILOT = train(pilot, ROOT, client=client)
print(PILOT, f"всего {(time.time() - STARTED) / 60:.1f} мин")

In [ ]:
PILOT = PILOT if "PILOT" in globals() else sorted((ROOT / "runs" / "ppo_pilot").glob("*_seed*"))[-1]
for r in [r for r in read_journal(PILOT / "steps.jsonl") if "step" in r]:
    t = r["seconds"]
    c, p2 = r["critic_before"], r["passes"][-1]
    print(f"шаг {r['step']}: сбор {t['generation']:.0f} с, награда {t['reward']:.0f}, проход 0 {t['pass0']:.0f}, "
          f"голова {t['head']:.0f}, политика {t['train']:.0f}, адаптер {t['adapter']:.0f} | пик {r['peak_memory_gb']:.1f} ГБ")
    print(f"   критик до обновления: MSE {c['all_mse']:.5f} против константы {c['all_mse_const']:.5f}, смещение "
          f"{c['all_bias']:+.4f} | второй проход: вне диапазона {p2['ratio_outside']:.3f}, отсечено {p2['clip_active']:.3f}, "
          f"ρ max {p2.get('ratio_max')} | KL(old‖new) {r['kl_old_new_k3']} | обновлений {r['updates']}")

## 5. Обучение

`RUN_DIR = None` — новый запуск (`runs/ppo/<дата_время>_seed<сид>`); путь к папке — продолжение после сбоя
с последнего полного состояния (каждые 5 шагов). Нужен готовый разогрев этого сида (`head.pt` в `WARMUP`).

In [ ]:
from presentation_rl.training.ppo import train

RUN_DIR = None  # например ROOT / "runs/ppo/20261119_2000_seed0" — продолжение
run = train(config, ROOT, run_dir=RUN_DIR, client=client)
print(run)